# Banco Río: perfiles de cartera y riesgo de crédito

Proyecto final de Programación Analítica II (Universidad de Montevideo, S2 2026). Propuesta A.

Fecha de corte de los datos: 30/06/2026. Originación: enero 2023 a marzo 2026.

## 1. EDA y calidad de datos

### 1.1 Carga de los tres archivos declarando tipos y fechas

Criterios de la carga:

- **Identificadores como texto** (`id_cliente`, `id_producto`): son etiquetas, no cantidades; nunca se suman ni se promedian.
- **Columnas de texto como `str`**, no como `category`: varias vienen escritas de distintas maneras ("Montevideo" / "montevideo"). Si se convierten a categoría ahora, cada variante quedaría como una categoría distinta. Se convierten después de estandarizarlas.
- **`ingreso_mensual` como texto**: viene en tres formatos ("34500.00", "$ 55.400", "29.100,00"). Si se lee como número, pandas falla o descarta valores. Se convierte en el punto 1.6.
- **Enteros con faltantes como `float64`**: `int64` no admite `NaN`.
- **Fechas con el formato declarado**: así pandas no adivina y un formato inesperado queda a la vista. Dos columnas de productos mezclan `AAAA-MM-DD` y `DD/MM/AAAA` y se convierten aparte.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

In [2]:
# Rutas: el notebook vive en notebooks/, los datos están una carpeta más arriba
CARPETA_DATOS = Path("..") / "datos"
RUTA_CLIENTES = CARPETA_DATOS / "clientes.csv"
RUTA_PRODUCTOS = CARPETA_DATOS / "productos.csv"
RUTA_TIPO_CAMBIO = CARPETA_DATOS / "tipo_cambio_mensual.csv"

FILAS_A_MOSTRAR = 5

# Formatos de fecha que aparecen en los archivos
FORMATO_ISO = "%Y-%m-%d"
FORMATO_DIA_MES_ANIO = "%d/%m/%Y"
FORMATO_MES = "%Y-%m"
PATRON_ISO = r"^\d{4}-\d{2}-\d{2}$"
PATRON_DIA_MES_ANIO = r"^\d{2}/\d{2}/\d{4}$"

TIPOS_CLIENTES = {
    "id_cliente": "str",
    "edad": "float64",  # tiene faltantes
    "sexo": "str",
    "estado_civil": "str",
    "cantidad_dependientes": "int64",
    "departamento": "str",
    "nivel_educativo": "str",
    "sector_empleo": "str",
    "antiguedad_laboral_meses": "float64",  # tiene faltantes
    "situacion_vivienda": "str",
    "ingreso_mensual": "str",  # tres formatos distintos, se convierte en 1.6
    "score_clearing": "float64",  # tiene faltantes
    "canal_originacion": "str",
    "segmento_comercial": "str",
    "tiene_cuenta_sueldo": "str",
}
FECHAS_CLIENTES = {"fecha_alta_cliente": FORMATO_ISO}

TIPOS_PRODUCTOS = {
    "id_producto": "str",
    "id_cliente": "str",  # texto, igual que en clientes, para que el merge coincida
    "tipo_producto": "str",
    "moneda": "str",
    "fecha_alta_producto": "str",  # dos formatos, se convierte aparte
    "plazo_meses": "int64",
    "tasa_nominal_anual": "float64",
    "score_originacion": "float64",  # tiene faltantes
    "monto_otorgado": "float64",
    "cuota_mensual": "float64",
    "limite_credito": "float64",
    "saldo_actual": "float64",
    "saldo_maximo_historico": "float64",
    "cuotas_pagas_en_termino": "int64",
    "veces_atraso_leve": "int64",
    "veces_atraso_medio": "int64",
    "veces_atraso_grave": "int64",
    "dias_atraso_acumulados": "int64",
    "fecha_primer_atraso_grave": "str",  # dos formatos, se convierte aparte
}
FECHAS_PRODUCTOS = {
    "fecha_ultima_actualizacion": FORMATO_ISO,
    "fecha_ultimo_reporte": FORMATO_DIA_MES_ANIO,
}
COLUMNAS_FECHA_DOS_FORMATOS = ["fecha_alta_producto", "fecha_primer_atraso_grave"]

TIPOS_TIPO_CAMBIO = {"tipo_cambio_usd": "float64", "valor_unidad_indexada": "float64"}
FECHAS_TIPO_CAMBIO = {"mes": FORMATO_MES}

# Categorías que debería tener cada columna según el enunciado (19 = departamentos
# de Uruguay). Sirve para comparar contra lo que trae el archivo en el punto 1.2.
CATEGORIAS_ESPERADAS = {
    "sexo": 2,
    "estado_civil": 5,
    "departamento": 19,
    "sector_empleo": 5,
    "situacion_vivienda": 4,
    "canal_originacion": 5,
    "segmento_comercial": 3,
    "tiene_cuenta_sueldo": 2,
    "tipo_producto": 6,
    "moneda": 3,
}

In [3]:
def cargar_tabla(ruta, tipos_columnas, formatos_fecha):
    """Lee un CSV declarando el tipo de cada columna y el formato de cada fecha."""
    return pd.read_csv(
        ruta,
        dtype=tipos_columnas,
        parse_dates=list(formatos_fecha),
        date_format=formatos_fecha,
    )


def convertir_fecha_dos_formatos(serie_texto):
    """Convierte a fecha una columna que mezcla AAAA-MM-DD y DD/MM/AAAA."""
    fecha_iso = pd.to_datetime(serie_texto, format=FORMATO_ISO, errors="coerce")
    fecha_dia_mes_anio = pd.to_datetime(
        serie_texto, format=FORMATO_DIA_MES_ANIO, errors="coerce"
    )
    return fecha_iso.fillna(fecha_dia_mes_anio)


def resumir_conversion_fecha(serie_texto, serie_fecha):
    """Cuenta las filas de cada formato y las que no se pudieron convertir."""
    filas_con_dato = serie_texto.notna().sum()
    filas_dia_mes_anio = serie_texto.str.match(PATRON_DIA_MES_ANIO).sum()
    return {
        "filas_con_dato": filas_con_dato,
        "formato_iso": serie_texto.str.match(PATRON_ISO).sum(),
        "formato_dia_mes_anio": filas_dia_mes_anio,
        "pct_dia_mes_anio": round(100 * filas_dia_mes_anio / filas_con_dato, 1),
        "fechas_no_convertidas": (serie_texto.notna() & serie_fecha.isna()).sum(),
    }


def resumir_columnas_fecha(tabla):
    """Devuelve el rango y los faltantes de cada columna de fecha de la tabla."""
    columnas_fecha = tabla.select_dtypes(include="datetime").columns
    return pd.DataFrame(
        {
            "fecha_minima": tabla[columnas_fecha].min(),
            "fecha_maxima": tabla[columnas_fecha].max(),
            "faltantes": tabla[columnas_fecha].isna().sum(),
        }
    )

In [4]:
clientes = cargar_tabla(
    ruta=RUTA_CLIENTES,
    tipos_columnas=TIPOS_CLIENTES,
    formatos_fecha=FECHAS_CLIENTES,
)
productos = cargar_tabla(
    ruta=RUTA_PRODUCTOS,
    tipos_columnas=TIPOS_PRODUCTOS,
    formatos_fecha=FECHAS_PRODUCTOS,
)
tipo_cambio = cargar_tabla(
    ruta=RUTA_TIPO_CAMBIO,
    tipos_columnas=TIPOS_TIPO_CAMBIO,
    formatos_fecha=FECHAS_TIPO_CAMBIO,
)

**Fechas con dos formatos.** Se convierte cada columna probando los dos formatos y se reporta cuántas filas venían en cada uno. Si alguna fecha quedara sin convertir (`fechas_no_convertidas` > 0) habría un tercer formato que no vimos.

In [5]:
# Modificación: texto -> fecha. Se reporta el % de filas que venía en DD/MM/AAAA
# y se verifica que ninguna fecha con dato se pierda en la conversión.
resumen_conversion = {}
for columna in COLUMNAS_FECHA_DOS_FORMATOS:
    fecha_convertida = convertir_fecha_dos_formatos(productos[columna])
    resumen_conversion[columna] = resumir_conversion_fecha(
        productos[columna], fecha_convertida
    )
    productos[columna] = fecha_convertida

display(pd.DataFrame.from_dict(resumen_conversion, orient="index"))

,filas_con_dato,formato_iso,formato_dia_mes_anio,pct_dia_mes_anio,fechas_no_convertidas
fecha_alta_producto,6191,5321,870,14.1,0
fecha_primer_atraso_grave,2438,2124,314,12.9,0


**Qué encontramos:** el 14,1% de las fechas de alta de productos (870 de 6.191) y el 12,9% de las fechas de primer atraso grave (314 de 2.438) venían en formato `DD/MM/AAAA`; el resto en `AAAA-MM-DD`.

**Evidencia:** las 8.629 fechas con dato se convirtieron y ninguna quedó sin convertir (`fechas_no_convertidas` = 0), así que no hay un tercer formato escondido. Los faltantes de `fecha_primer_atraso_grave` ya venían vacíos en el archivo: no los generó la conversión.

**Qué sugiere:** si se hubiera dejado que pandas adivinara el formato, una fecha como 03/04/2025 podía leerse como 4 de marzo en lugar de 3 de abril y mover créditos de trimestre, que es justamente el corte que usamos para ver si la cartera mejora o empeora.

**Primeras filas de cada tabla.**

In [6]:
display(clientes.head(FILAS_A_MOSTRAR))
display(productos.head(FILAS_A_MOSTRAR))
display(tipo_cambio.head(FILAS_A_MOSTRAR))

,id_cliente,fecha_alta_cliente,edad,sexo,estado_civil,cantidad_dependientes,departamento,nivel_educativo,sector_empleo,antiguedad_laboral_meses,situacion_vivienda,ingreso_mensual,score_clearing,canal_originacion,segmento_comercial,tiene_cuenta_sueldo
0,7807176,2021-10-09,70.0,F,Soltero/a,1,Montevideo,Secundaria completa,Jubilado,0.0,Propia con hipoteca,34500.00,603.0,Sucursal,Masivo,No
1,3180655,2022-07-11,52.0,M,Soltero/a,1,Montevideo,Secundaria completa,Público,51.0,Propia,$ 55.400,597.0,Comercio adherido,Masivo,No
2,3028040,2020-11-08,42.0,F,Casado/a,0,Maldonado,Secundaria completa,Independiente,6.0,Alquiler,100700,546.0,Sucursal,Preferente,No
3,6712544,2018-08-02,59.0,F,Soltero/a,1,Tacuarembó,Secundaria completa,Privado,42.0,Alquiler,90500.00,619.0,Sucursal,Preferente,No
4,8244977,2022-06-27,47.0,F,Unión libre,3,Canelones,Terciaria completa,Privado,50.0,Propia,"59.500,00",689.0,Web,Masivo,Si


,id_producto,id_cliente,tipo_producto,moneda,fecha_alta_producto,plazo_meses,tasa_nominal_anual,score_originacion,monto_otorgado,cuota_mensual,...,saldo_actual,saldo_maximo_historico,cuotas_pagas_en_termino,veces_atraso_leve,veces_atraso_medio,veces_atraso_grave,dias_atraso_acumulados,fecha_primer_atraso_grave,fecha_ultima_actualizacion,fecha_ultimo_reporte
0,900023,10817904,Adelanto de sueldo,UYU,2023-03-22,10,0.7533,693.0,42924.53,5908.98,...,0.00,NaN,10,0,0,0,0,NaT,2026-06-19,2026-05-22
1,900035,10504807,Tarjeta de crédito,UYU,2024-08-27,12,0.9147,601.0,250835.46,32636.20,...,0.00,304682.81,6,3,2,1,121,2025-05-19,2026-06-17,NaT
2,505812,1113415,Préstamo automotor,USD,2024-02-24,31,0.1052,690.0,24043.55,889.12,...,2262.21,27384.35,19,4,2,1,213,2025-11-10,2026-06-13,2026-04-01
3,500952,4857398,Préstamo automotor,USD,2024-10-26,43,0.1493,610.0,22294.55,672.58,...,13258.66,24836.47,16,2,2,0,104,NaT,2026-06-08,2026-03-21
4,505792,6176407,Adelanto de sueldo,UYU,2023-03-21,6,1.0948,722.0,80960.83,18114.34,...,0.00,81084.13,5,1,0,0,19,NaT,2026-06-06,2026-04-10


,mes,tipo_cambio_usd,valor_unidad_indexada
0,2023-01-01,39.037,5.6589
1,2023-02-01,38.729,5.6993
2,2023-03-01,39.448,5.7434
3,2023-04-01,40.052,5.7729
4,2023-05-01,40.124,5.7937


**Verificación de la carga.** Se controla que cada columna tenga el tipo declarado y que las fechas caigan en rangos razonables: altas de productos entre enero 2023 y marzo 2026, nada posterior al corte del 30/06/2026 y tipo de cambio para todos los meses de otorgamiento.

In [7]:
tipos_cargados = pd.concat(
    {
        "clientes": clientes.dtypes.astype(str),
        "productos": productos.dtypes.astype(str),
        "tipo_cambio": tipo_cambio.dtypes.astype(str),
    }
)
display(tipos_cargados.value_counts().to_frame("cantidad_columnas"))

,cantidad_columnas
str,15
float64,12
int64,7
datetime64[us],6


In [8]:
display(resumir_columnas_fecha(clientes))
display(resumir_columnas_fecha(productos))
display(resumir_columnas_fecha(tipo_cambio))

,fecha_minima,fecha_maxima,faltantes
fecha_alta_cliente,2013-02-24,2022-12-02,0


,fecha_minima,fecha_maxima,faltantes
fecha_alta_producto,2023-01-01,2026-03-30,0
fecha_primer_atraso_grave,2023-03-28,2026-06-24,3753
fecha_ultima_actualizacion,2026-05-17,2026-06-30,0
fecha_ultimo_reporte,2026-02-17,2026-06-30,567


,fecha_minima,fecha_maxima,faltantes
mes,2023-01-01,2026-06-01,0


**Qué encontramos:** las 40 columnas quedaron con el tipo declarado (16 de clientes + 21 de productos + 3 de tipo de cambio = 15 de texto, 12 decimales, 7 enteras y 6 fechas) y las fechas caen donde deben: las altas de productos van del 01/01/2023 al 30/03/2026, igual que el período de originación del enunciado, y ninguna fecha pasa del corte del 30/06/2026.

**Evidencia:** los clientes se dieron de alta entre 2013 y diciembre de 2022, antes de su primer producto; el tipo de cambio cubre de enero 2023 a junio 2026 (42 meses), así que todos los meses de otorgamiento tienen cotización para pasar los montos a pesos.

**Qué sugiere y qué falta:** la carga es confiable para seguir. Quedan dos grupos de vacíos para el punto 1.4. Los 3.753 de `fecha_primer_atraso_grave` no son datos perdidos: según el enunciado la fecha queda "vacía si nunca pasó", y eso hay que confirmarlo cruzándola con `veces_atraso_grave`. Los 567 de `fecha_ultimo_reporte` no tienen explicación en el enunciado: hay que ver si se concentran en algún grupo.

### 1.2 Resumen de cada tabla

Antes de limpiar nada se saca una "foto" de cada tabla: cuántas filas y columnas tiene, cuántas categorías distintas hay en cada columna de texto (comparadas con las que debería tener según el enunciado) y la descripción estadística de las columnas numéricas. Los tipos de cada columna ya se verificaron en el punto 1.1.

Esta foto no corrige nada: sirve para ver dónde están los problemas y medir después cuánto cambió cada cosa al limpiar.

In [9]:
def resumir_dimensiones(tablas):
    """Devuelve la cantidad de filas y columnas de cada tabla."""
    return pd.DataFrame(
        {
            nombre: {"filas": tabla.shape[0], "columnas": tabla.shape[1]}
            for nombre, tabla in tablas.items()
        }
    ).T


def contar_categorias(tabla, categorias_esperadas):
    """Cuenta los valores distintos de cada columna de texto contra los esperados."""
    columnas_texto = tabla.select_dtypes(include="str").columns
    resumen = tabla[columnas_texto].nunique().to_frame("categorias_en_archivo")
    resumen["categorias_esperadas"] = (
        resumen.index.to_series().map(categorias_esperadas).astype("Int64")
    )
    return resumen.sort_values("categorias_en_archivo", ascending=False)


def describir_continuas(tabla):
    """Devuelve la descripción estadística de las columnas numéricas."""
    return tabla.select_dtypes(include="number").describe().T.round(2)

In [10]:
# Junto las tres tablas en un diccionario para resumirlas con la misma función
tablas_del_banco = {
    "clientes": clientes,
    "productos": productos,
    "tipo_cambio": tipo_cambio,
}
display(resumir_dimensiones(tablas_del_banco))

,filas,columnas
clientes,3235,16
productos,6191,21
tipo_cambio,42,3


**Qué encontramos:** clientes tiene 3.235 filas y 16 columnas, productos 6.191 filas y 21 columnas, y tipo de cambio 42 filas y 3 columnas. Según el enunciado, una fila de clientes es un cliente, una fila de productos es un producto de crédito (un cliente puede tener varios) y una fila de tipo de cambio es un mes (de enero 2023 a junio 2026).

**Evidencia:** en clientes debería haber una fila por cliente, pero hay 3.235 filas y solo 3.200 `id_cliente` distintos (tabla de abajo); en productos hay 6.191 filas y 6.051 `id_producto` distintos. Sobran 35 filas en clientes y 140 en productos.

**Qué sugiere:** hay filas repetidas en las dos tablas. Si no se resuelven antes del merge, un cliente repetido duplicaría sus productos y la mora quedaría mal calculada. Se separan en el punto 1.3 (duplicados exactos contra id repetido con datos distintos).

#### Clientes

In [11]:
display(contar_categorias(clientes, CATEGORIAS_ESPERADAS))
display(describir_continuas(clientes))

,categorias_en_archivo,categorias_esperadas
id_cliente,3200,<NA>
ingreso_mensual,2083,<NA>
departamento,55,19
canal_originacion,10,5
sector_empleo,10,5
situacion_vivienda,7,4
nivel_educativo,6,<NA>
estado_civil,5,5
segmento_comercial,3,3
sexo,2,2


,count,mean,std,min,25%,50%,75%,max
edad,3225.0,55.52,15.57,0.0,46.0,54.0,63.0,202.0
cantidad_dependientes,3235.0,1.11,1.06,0.0,0.0,1.0,2.0,6.0
antiguedad_laboral_meses,3018.0,55.19,74.27,-33.0,9.0,37.0,80.0,771.0
score_clearing,3094.0,647.62,125.61,0.0,595.0,659.0,719.0,950.0


**Qué encontramos:** 4 columnas de texto tienen más categorías que las que deberían: `departamento` tiene 55 cuando Uruguay tiene 19, `canal_originacion` y `sector_empleo` tienen 10 cuando deberían ser 5, y `situacion_vivienda` tiene 7 cuando deberían ser 4. En las numéricas hay valores imposibles: edad mínima 0 y máxima 202, antigüedad laboral mínima de -33 meses y `score_clearing` mínimo de 0 cuando la escala va de 300 a 950.

**Evidencia:** conteo de valores distintos sobre las 3.235 filas y `describe` de las 4 columnas numéricas. `ingreso_mensual` no aparece en el `describe` porque se leyó como texto (sus 2.083 valores distintos vienen en tres formatos).

**Qué sugiere:** las categorías de más son la misma categoría escrita de varias maneras ("Montevideo" / "montevideo"), y se estandarizan en el punto 1.5 junto con los valores imposibles. Falta saber si esos valores imposibles son errores de carga o códigos que el sistema usa para decir "no sé".

#### Productos

In [12]:
display(contar_categorias(productos, CATEGORIAS_ESPERADAS))
display(describir_continuas(productos))

,categorias_en_archivo,categorias_esperadas
id_producto,6051,<NA>
id_cliente,3296,<NA>
tipo_producto,10,6
moneda,3,3


,count,mean,std,min,25%,50%,75%,max
plazo_meses,6191.0,34.48,65.25,0.00,12.00,13.00,37.00,999.00
tasa_nominal_anual,6191.0,4.42,16.28,0.06,0.53,0.71,0.87,109.34
score_originacion,6035.0,637.59,115.81,0.00,580.00,648.00,708.00,950.00
monto_otorgado,6191.0,219583.18,800567.16,-460190.39,54919.57,121710.11,235695.05,33887347.00
cuota_mensual,6191.0,15988.57,18569.72,43.09,4938.16,10525.49,20772.67,390273.93
limite_credito,5351.0,273129.92,328559.06,2279.95,77868.98,173533.56,339680.15,4864991.70
saldo_actual,6191.0,67827.66,170196.52,0.00,0.00,0.00,55920.67,1759940.54
saldo_maximo_historico,5892.0,212979.95,253229.95,1855.52,61751.19,137521.40,263475.91,3894419.72
cuotas_pagas_en_termino,6191.0,11.55,8.16,0.00,6.00,10.00,15.00,41.00
veces_atraso_leve,6191.0,1.37,1.38,0.00,0.00,1.00,2.00,12.00


**Qué encontramos:** `tipo_producto` tiene 10 categorías cuando el enunciado lista 6. En las numéricas hay cuatro señales de problemas: `plazo_meses` llega a 999 y tiene mínimo 0; `tasa_nominal_anual` tiene promedio 4,42 pero mediana 0,71; `monto_otorgado` tiene montos negativos (mínimo -460.190); y `dias_atraso_acumulados` tiene mínimo -999.

**Evidencia:** conteo de valores distintos y `describe` de las 13 columnas numéricas sobre las 6.191 filas. En la tasa, el promedio es 6 veces la mediana: la mayoría de las tasas están entre 0,53 y 0,87 (fracción, o sea 53% a 87%), pero unas pocas llegan a 109,34, que solo tiene sentido si están escritas en porcentaje.

**Qué sugiere:** la tasa es la candidata a columna con dos unidades (punto 1.6), y el 999 y el -999 parecen códigos de "no sé" (punto 1.5). Además, `monto_otorgado` y `cuota_mensual` mezclan pesos, dólares y UI, así que su promedio todavía no significa nada: hay que pasarlos a pesos con el tipo de cambio del mes de otorgamiento (punto 2.2). Otra cosa a mirar: productos tiene 3.296 `id_cliente` distintos y clientes solo 3.200, así que hay productos de clientes que no están en la tabla de clientes.

#### Tipo de cambio

In [13]:
display(describir_continuas(tipo_cambio))

,count,mean,std,min,25%,50%,75%,max
tipo_cambio_usd,42.0,42.04,2.08,38.73,40.72,41.33,44.40,45.52
valor_unidad_indexada,42.0,6.19,0.32,5.66,5.89,6.19,6.47,6.75


**Qué encontramos:** el dólar se movió entre $38,73 y $45,52 y la unidad indexada entre $5,66 y $6,75 en los 42 meses, sin faltantes ni valores raros.

**Evidencia:** `describe` de las 2 columnas sobre las 42 filas. La tabla no tiene columnas de texto, por eso no lleva conteo de categorías.

**Qué sugiere:** la tabla está limpia y se puede usar tal cual para convertir montos. Entre el mínimo y el máximo del dólar hay más de 17% de diferencia, así que usar un tipo de cambio fijo en lugar del del mes de otorgamiento movería bastante los montos en pesos.